# Förberedelser

## Importera

In [ ]:
import SPARQLWrapper as sparql
import pandas as pd

In [ ]:
endpoint = "https://libris.kb.se/sparql"

# Analys

## Digitiseringar
Den digitiserade versionen av en fysisk resurs. 

I dagsläget finns en hänvisning, som lokal entitet, till det fysiska originalet i otherPhysicalFormat > describedBy > controlNumber .

Målet är att byta ut den lokala entiteten mot en länakd entitet i reproductionOf.

In [ ]:
query = '''

PREFIX marc: <https://id.kb.se/marc/>
PREFIX : <https://id.kb.se/vocab/>

SELECT  ?instance ?recordControlNumber ?localControlNumber ?marcDisplayText WHERE {

	?record :mainEntity ?instance ;
    :controlNumber ?recordControlNumber .
    
  ?instance a :DigitalResource ;
    :otherPhysicalFormat ?localEntity .
    # :sameAs ?recordControlNumber .

  ?localEntity :describedBy [ a :Record;
     :controlNumber ?localControlNumber ] ;
     marc:displayText ?marcDisplayText .

  FILTER(REGEX(?localControlNumber, "^[0-9a-z]+$"))

  FILTER isBLANK(?localEntity) .

  FILTER(regex(?marcDisplayText, "(originalversion|orginalversion)", "i"))

}

'''

digitizations = sparql.get_sparql_dataframe(endpoint, query)
digitizations.info()
digitizations.head(1)

### Finns det digitiseringar som hänviar till flera original? 🤯

In [ ]:
repros_with_multiple_originals = digitizations[digitizations.duplicated(subset=["instance"], keep=False)]
repros_with_multiple_originals.value_counts("instance")

Droppa dessa, för att istället hantera manuellt.

In [ ]:
digitizations.drop_duplicates(subset="instance", keep=False, inplace=True)
digitizations.info()

## Original
Den fysiska resurs som digitiserats.

In [ ]:
query = '''

PREFIX marc: <https://id.kb.se/marc/>
PREFIX : <https://id.kb.se/vocab/>

SELECT ?instance ?recordControlNumber ?localControlNumber ?marcDisplayText WHERE {

	?record :mainEntity ?instance ;
    :controlNumber ?recordControlNumber .

  ?instance a :PhysicalResource ;
    :otherPhysicalFormat ?localEntity .
    # :sameAs ?recordControlNumber .

  FILTER isBLANK(?localEntity) .

  ?localEntity :describedBy [ a :Record;
     :controlNumber ?localControlNumber ] ;
     marc:displayText ?marcDisplayText .

  FILTER(REGEX(?localControlNumber, "^[0-9a-z]+$"))
  FILTER(regex(?marcDisplayText, "(^digit.*version)", "i"))
  FILTER(!regex(?note, "(del|1969)", "i"))

}

'''

originals = sparql.get_sparql_dataframe(endpoint, query)
originals.info()
originals.head(1)

### Finns det original som pekar mot flera digitiseringar?



In [ ]:
originals_with_multiple_repros = originals[originals.duplicated(subset=["instance"], keep=False)]
originals_with_multiple_repros.info()
originals_with_multiple_repros.value_counts("instance")

## Jämförelse

### Hita och skriv ut de ömsesidigt matchande

In [ ]:
mutual_matches = digitizations.merge(
    originals,
    how="inner",
    left_on=["localControlNumber", "recordControlNumber"],
    right_on=["recordControlNumber", "localControlNumber"],
    suffixes=("_digital", "_physical")
)

# Helt identiska rader kan droppas
mutual_matches.drop_duplicates(inplace=True)

mutual_matches.info()
mutual_matches.head(5)

mutual_matches_to_print = mutual_matches.copy()

LIBRIS_ID_REGEX = r"https://libris(?:-(?:qa|stg))?\.kb\.se/([a-zA-Z0-9]+)#it"

mutual_matches_to_print["instance_digital"] = mutual_matches_to_print["instance_digital"].str.extract(LIBRIS_ID_REGEX, expand=False)
mutual_matches_to_print["instance_physical"] = mutual_matches_to_print["instance_physical"].str.extract(LIBRIS_ID_REGEX, expand=False)
mutual_matches_to_print.to_csv("mutual_matches_ids.tsv", sep="\t", index=False)

### Djupdykning

In [ ]:
# Vilka av de ömsesidigt matchande originalen har flera reproduktioner?
mutual_matches[mutual_matches.duplicated(subset=["instance_physical"], keep=False)].sort_values("instance_physical")

In [ ]:
# Vilka reproduktioner har flera original?
mutual_matches[mutual_matches.duplicated(subset=["instance_digital"], keep=False)]

#### Ensidiga matcher

In [ ]:
digi_to_physical_matches = digitizations.merge(
    originals,
    how="inner",
    left_on=["localControlNumber"],
    right_on=["recordControlNumber"],
    suffixes=("_digital", "_physical")
)

digi_to_physical_matches.drop_duplicates(inplace=True)

digi_to_physical_matches.info()

In [ ]:
physical_to_digi_matches = digitizations.merge(
    originals,
    how="inner",
    left_on=["recordControlNumber"],
    right_on=["localControlNumber"],
    suffixes=("_digital", "_physical")
)

physical_to_digi_matches.drop_duplicates(inplace=True)

physical_to_digi_matches.info()

In [ ]:
cols = physical_to_digi_matches.columns

non_mutual_matches = pd.concat(
    [physical_to_digi_matches.assign(source="physical_to_digi"),
     digi_to_physical_matches.assign(source="digi_to_physical")]
    ).drop_duplicates(subset=cols, keep=False)

non_mutual_matches.info()
non_mutual_matches.head(1)

- Rader med source "physical_to_digi" representerar fall där en fysisk instans pekar på en digital instans, som inte pekar tillbaka.

- Rader med source "digi_to_physical" representerar fall där en digital instans pekar på en fysisk instans, som inte pekar tillbaka.

In [ ]:
non_mutual_matches.value_counts("source")

In [ ]:
non_mutual_matches.value_counts(["instance_digital", "source"])

In [ ]:
non_mutual_matches.to_csv("non_mutual_matches.tsv", sep="\t")